# Harmony reference workflow on Dataset 0 (GSE194122)

This Google Colab notebook is a detailed **reference example** showing how an external Harmony workflow can be benchmarked with scRareBench. The method remains user-side.

Flow:
1. install the pinned scRareBench release and the notebook-owned `harmonypy==2.0.0` dependency;
2. load the curated GSE194122 benchmark;
3. perform Harmony-specific preprocessing from raw counts;
4. run Harmony using PCA coordinates and `BATCH` only;
5. verify latent/cell alignment;
6. run scRareBench metrics and reporting;
7. create static HTML, interactive HTML, PDF, and ZIP artifacts.

Harmony integration itself does not require a GPU. A high-memory Colab runtime is useful for scaling/PCA on 4,000 HVGs. `celltype` is never supplied to Harmony and is used only during benchmarking.

> **Validation note:** this notebook was tested on Google Colab runtime 2026.07. An optional, fully commented compatibility-check cell is included for diagnostics only. Exact NumPy/PyTorch/JAX version matching is **not required** to run the notebook.


## 1. Install the pinned scRareBench release

This notebook does not require a local source ZIP. It first bootstraps the pinned scRareBench release without dependencies, then calls the generic `scrarebench.runtime.setup_runtime()` helper. The method dependency is declared explicitly by this notebook; scRareBench itself does not know or register the method. The runtime helper checks dependency health, preserves the scientific ABI-sensitive packages already present in the current environment, and runs fresh-process import smoke tests. The separate compatibility-check cell is optional and disabled by default.


In [ ]:
SCRAREBENCH_GITHUB = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@v0.10.5"
METHOD = "harmony"

# To pin a release/commit, for development only, replace the release tag with the development branch explicitly.


In [ ]:
# OPTIONAL: validate this runtime against the environment used for release testing.
# This check is informational only and is NOT required to run scRareBench or this notebook.
# Leave this cell unchanged to skip the check. Uncomment the lines below if you want to compare
# your current environment with the Google Colab 2026.07 runtime used during validation.
# A different compatible runtime may still work correctly.
#
# import sys
# from importlib import metadata as _runtime_metadata
#
# _EXPECTED_COLAB_ANCHORS = {
#     "numpy": "2.0.2",
#     "torch": "2.11.0",
#     "jax": "0.7.2",
# }
# _runtime_mismatches = []
# for _package, _expected in _EXPECTED_COLAB_ANCHORS.items():
#     try:
#         _observed = _runtime_metadata.version(_package)
#     except _runtime_metadata.PackageNotFoundError:
#         _observed = "not installed"
#     if _observed != _expected:
#         _runtime_mismatches.append(
#             f"{_package}: validated {_expected}, current {_observed}"
#         )
#
# if _runtime_mismatches:
#     print("Runtime differs from the Colab 2026.07 validation environment:")
#     for _item in _runtime_mismatches:
#         print(" -", _item)
# else:
#     print("Runtime matches the documented Colab 2026.07 validation anchors.")


In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

# Bootstrap only the lightweight scRareBench package code from the pinned release.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCRAREBENCH_GITHUB
])

from scrarebench.runtime import print_install_report, setup_runtime

# setup_runtime preserves ABI-sensitive scientific packages already present in the
# current environment. The optional compatibility-check cell above is informational
# only; matching the exact Colab 2026.07 anchor versions is not required.
install_report = setup_runtime(
    extra_requirements=('harmonypy==2.0.0',),
    extra_imports=('harmonypy',),
    quiet=False,
)
print_install_report(install_report)

# Import only after dependency validation and the fresh-process smoke test pass.
import scrarebench as _scrarebench_install_check
import scib_metrics as _scib_metrics_install_check
print("scrarebench import path:", Path(_scrarebench_install_check.__file__).resolve())
print("scrarebench version:", _scrarebench_install_check.__version__)
print("scib-metrics version:", getattr(_scib_metrics_install_check, "__version__", "installed"))
import harmonypy as _method_install_check
print("harmonypy version:", getattr(_method_install_check, "__version__", "2.0.0"))


## 2. Imports and environment information


In [ ]:
from __future__ import annotations

from pathlib import Path
import gc
import hashlib
import json
import os
import platform
import sys
import importlib
import shutil
import time
import warnings

import anndata as ad
import harmonypy as hm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import yaml


import scrarebench
from IPython.display import HTML, display

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("scRareBench:", scrarebench.__version__)
print("scIB runtime compatibility: automatic (pandas 2/3 safe)")
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("harmonypy:", getattr(hm, "__version__", "2.0.0"))
print("Harmony runs on CPU; GPU is not required for the integration step.")


## 3. Harmony and benchmark configuration


In [ ]:
# Runtime paths
WORK_DIR = Path("/content/scrarebench_harmony_run")
DATA_DIR = WORK_DIR / "data"
CACHE_DIR = DATA_DIR / "cache"
HARMONY_CACHE_DIR = WORK_DIR / "harmony_cache"
RESULTS_DIR = WORK_DIR / "results" / "Harmony"
ARTIFACT_DIR = WORK_DIR / "deliverable"

for directory in (WORK_DIR, DATA_DIR, CACHE_DIR, HARMONY_CACHE_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

BATCH_KEY = "BATCH"
LABEL_KEY = "celltype"
COUNTS_LAYER = "counts"
PCA_KEY = "X_pca"
LATENT_KEY = "X_pca_harmony"
METHOD_NAME = "Harmony"

# Harmony-specific preprocessing
SEED = 42
N_HVG = 4000
NORMALIZE_TARGET_SUM = 10_000.0
SCALE_DATA = True
SCALE_MAX_VALUE = 10.0
N_PCS = 50
PCA_SOLVER = "arpack"

# Harmony 2.0.0 settings kept close to the documented API defaults.
HARMONY_THETA = 2.0
HARMONY_LAMBDA = None          # None => dynamic lambda estimation in harmonypy 2.0
HARMONY_SIGMA = 0.1
HARMONY_N_CLUSTERS = None     # None => min(round(N/30), 100)
HARMONY_TAU = 0.0
HARMONY_BLOCK_SIZE = 0.05
HARMONY_MAX_ITER = 10
HARMONY_MAX_ITER_KMEANS = 4
HARMONY_EPSILON_CLUSTER = 1e-3
HARMONY_EPSILON_HARMONY = 1e-2
HARMONY_ALPHA = 0.2
HARMONY_BATCH_PROP_CUTOFF = 1e-5
HARMONY_NCORES = 0            # 0 => use all BLAS threads available to harmonypy
HARMONY_VERBOSE = True
REUSE_MATCHING_HARMONY_LATENT = True

# Package-controlled clustering for the paper/rare-cell layer
N_NEIGHBORS = 15
REFERENCE_RESOLUTION = 1.0
RESOLUTION_SWEEP = (1.0,)
DISTANCE_METRIC = "euclidean"

# standard scIB-compatible layer
RUN_SCIB = True
SCIB_N_HVG = 4000
SCIB_REFERENCE_N_PCS = 50
SCIB_N_JOBS = 1
SCIB_PROGRESS_BAR = True
SCIB_INCLUDE_SILHOUETTE_BATCH = True
SCIB_REQUIRE_SUCCESS = True

FORCE_REBUILD_DATASET = False
GENERATE_UMAP = True
DOWNLOAD_RESULT_ZIP = True
DOWNLOAD_STANDALONE_REPORT = False
GENERATE_INTERACTIVE_REPORT = True
GENERATE_PDF_REPORT = True
INCLUDE_LATENT_IN_BUNDLE = True

# Interactive dashboard sections; all are enabled by default.
# Disabling a section also removes its payload from the HTML to reduce file size.
HTML_INCLUDE_OVERVIEW = True
HTML_INCLUDE_METRICS = True
HTML_INCLUDE_SCIB = True
HTML_INCLUDE_RARE = True
HTML_INCLUDE_RARE_UMAP = True
HTML_INCLUDE_RARE_HEATMAPS = True
HTML_INCLUDE_RARE_SCENARIO_ANALYSIS = True
HTML_INCLUDE_UMAP = True
HTML_INCLUDE_SANKEY = True
HTML_INCLUDE_REPRODUCIBILITY = True
HTML_INCLUDE_STATIC_FIGURES = True
HTML_INCLUDE_CELL_IDS = True

print("Work directory:", WORK_DIR)


## 4. Load the curated GSE194122 benchmark


In [ ]:
from scrarebench import load_dataset

benchmark_h5ad = DATA_DIR / "gse194122_paper_main.h5ad"

# Dataset selector 0 == "gse194122": download original source when needed,
# apply only the benchmark-specific cell subsetting + six-scenario annotation,
# and return the 89,199-cell AnnData. No Harmony preprocessing happens here.
adata = load_dataset(
    0,
    DATA_DIR,
    force_download=False,
    force_rebuild=FORCE_REBUILD_DATASET,
    strict_expected_counts=True,
)

manifest_path = benchmark_h5ad.with_suffix(".manifest.json")
dataset_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

assert adata.n_obs == 89_199, adata.n_obs
assert BATCH_KEY in adata.obs
assert LABEL_KEY in adata.obs
assert adata.obs_names.is_unique

print(adata)
print("Cells:", adata.n_obs)
print("Genes/features:", adata.n_vars)
print("Batches:", adata.obs[BATCH_KEY].nunique())
print("Cell types:", adata.obs[LABEL_KEY].nunique())
print("Cell-order hash:", dataset_manifest["cell_order_sha256"])

distribution_path = benchmark_h5ad.with_suffix(".distribution.csv")
display(pd.read_csv(distribution_path))


## 5. Harmony-specific preprocessing

This section belongs to the **user method**, not to the scRareBench dataset loader. It selects RNA/GEX features, validates raw counts, selects 4,000 batch-aware Seurat-v3 HVGs, and creates a method-specific AnnData while preserving benchmark cell order. Normalization, `log1p`, scaling, PCA, and Harmony are executed only when a matching latent cache is unavailable.


In [ ]:
def sampled_count_diagnostics(matrix, max_values: int = 100_000, seed: int = 42):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    values = np.asarray(values)
    if len(values) > max_values:
        rng = np.random.default_rng(seed)
        values = values[rng.choice(len(values), size=max_values, replace=False)]
    values = values.astype(float, copy=False)
    finite = values[np.isfinite(values)]
    diagnostics = {
        "sampled_values": len(values),
        "finite_fraction": float(np.mean(np.isfinite(values))) if len(values) else 1.0,
        "minimum": float(np.min(finite)) if len(finite) else 0.0,
        "maximum": float(np.max(finite)) if len(finite) else 0.0,
        "nonnegative": bool(len(finite) == 0 or finite.min() >= 0),
        "integer_like": bool(len(finite) == 0 or np.allclose(finite, np.rint(finite), atol=1e-6)),
    }
    return diagnostics

# RNA/GEX feature selection
if "feature_types" in adata.var.columns:
    feature_types = adata.var["feature_types"].astype(str).str.upper()
    rna_mask = feature_types.eq("GEX").to_numpy()
    if rna_mask.sum() == 0:
        raise ValueError("No GEX features were found in adata.var['feature_types'].")
    adata_rna = adata[:, rna_mask].copy() if rna_mask.sum() != adata.n_vars else adata
else:
    adata_rna = adata

adata_rna.var_names_make_unique()

if COUNTS_LAYER not in adata_rna.layers:
    raise KeyError(
        f"Raw-count layer {COUNTS_LAYER!r} is missing. Available layers: {list(adata_rna.layers)}"
    )

count_diagnostics = sampled_count_diagnostics(adata_rna.layers[COUNTS_LAYER], seed=SEED)
print("Count diagnostics:", count_diagnostics)
if not count_diagnostics["nonnegative"] or not count_diagnostics["integer_like"]:
    raise ValueError("The selected matrix does not look like nonnegative integer raw counts.")

# Batch-aware HVG selection on raw counts
sc.pp.highly_variable_genes(
    adata_rna,
    layer=COUNTS_LAYER,
    flavor="seurat_v3",
    n_top_genes=min(N_HVG, adata_rna.n_vars),
    batch_key=BATCH_KEY,
    subset=False,
)

hvg_mask = adata_rna.var["highly_variable"].fillna(False).to_numpy()
hvg_names = adata_rna.var_names[hvg_mask]
if len(hvg_names) == 0:
    raise RuntimeError("No HVGs were selected.")

counts_hvg = adata_rna.layers[COUNTS_LAYER][:, hvg_mask]
if sp.issparse(counts_hvg):
    counts_hvg = counts_hvg.tocsr().astype(np.float32)
else:
    counts_hvg = np.asarray(counts_hvg, dtype=np.float32)

adata_harmony = ad.AnnData(
    X=counts_hvg.copy(),
    obs=adata_rna.obs.copy(),
    var=adata_rna.var.loc[hvg_names].copy(),
)
adata_harmony.var_names_make_unique()

assert np.array_equal(adata_harmony.obs_names.astype(str), adata.obs_names.astype(str))
assert adata_harmony.n_vars == min(N_HVG, adata_rna.n_vars)

print(adata_harmony)
print("Selected HVGs:", adata_harmony.n_vars)
print("Cell order preserved:", np.array_equal(adata_harmony.obs_names.astype(str), adata.obs_names.astype(str)))


## 6. PCA and Harmony

The method-side pipeline is:

`raw counts -> normalize_total -> log1p -> scale -> PCA -> Harmony`

Harmony receives only the PCA representation and `BATCH`; reference cell labels are not used by the integration method. A cached latent is reused only when its manifest matches the current cell order, HVGs, batch labels, and method configuration.


In [ ]:
def hash_strings(values) -> str:
    digest = hashlib.sha256()
    for value in values:
        digest.update(str(value).encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()

harmony_config = {
    "seed": SEED,
    "n_hvg": N_HVG,
    "normalize_target_sum": NORMALIZE_TARGET_SUM,
    "scale_data": SCALE_DATA,
    "scale_max_value": SCALE_MAX_VALUE,
    "n_pcs": N_PCS,
    "pca_solver": PCA_SOLVER,
    "batch_key": BATCH_KEY,
    "theta": HARMONY_THETA,
    "lambda": HARMONY_LAMBDA,
    "sigma": HARMONY_SIGMA,
    "n_clusters": HARMONY_N_CLUSTERS,
    "tau": HARMONY_TAU,
    "block_size": HARMONY_BLOCK_SIZE,
    "max_iter_harmony": HARMONY_MAX_ITER,
    "max_iter_kmeans": HARMONY_MAX_ITER_KMEANS,
    "epsilon_cluster": HARMONY_EPSILON_CLUSTER,
    "epsilon_harmony": HARMONY_EPSILON_HARMONY,
    "alpha": HARMONY_ALPHA,
    "batch_prop_cutoff": HARMONY_BATCH_PROP_CUTOFF,
    "ncores": HARMONY_NCORES,
}

run_manifest = {
    "cell_hash": hash_strings(adata_harmony.obs_names),
    "gene_hash": hash_strings(adata_harmony.var_names),
    "batch_hash": hash_strings(adata_harmony.obs[BATCH_KEY].astype(str)),
    "n_obs": adata_harmony.n_obs,
    "n_vars": adata_harmony.n_vars,
    "config": harmony_config,
    "harmonypy_version": getattr(hm, "__version__", "2.0.0"),
    "scanpy_version": sc.__version__,
}

manifest_path = HARMONY_CACHE_DIR / "run_manifest.json"
cached_latent_path = HARMONY_CACHE_DIR / "Harmony_latent.npy"
cached_history_path = HARMONY_CACHE_DIR / "Harmony_objective_history.csv"

can_reuse = False
if REUSE_MATCHING_HARMONY_LATENT and manifest_path.exists() and cached_latent_path.exists():
    try:
        can_reuse = json.loads(manifest_path.read_text(encoding="utf-8")) == run_manifest
    except Exception:
        can_reuse = False

np.random.seed(SEED)

if can_reuse:
    print("Loading matching cached Harmony latent:", cached_latent_path)
    X_harmony = np.load(cached_latent_path, allow_pickle=False).astype(np.float32, copy=False)
    harmony_seconds = 0.0
    if cached_history_path.exists():
        history_df = pd.read_csv(cached_history_path)
    else:
        history_df = pd.DataFrame()
else:
    print(f"Preparing Harmony input on {adata_harmony.n_obs:,} cells and {adata_harmony.n_vars:,} HVGs")

    # Method-specific preprocessing for Harmony.
    sc.pp.normalize_total(adata_harmony, target_sum=NORMALIZE_TARGET_SUM)
    sc.pp.log1p(adata_harmony)
    if SCALE_DATA:
        # Scaling can densify X; high-RAM Colab is recommended for this benchmark.
        sc.pp.scale(adata_harmony, max_value=SCALE_MAX_VALUE)

    n_pcs_effective = min(N_PCS, adata_harmony.n_vars - 1, adata_harmony.n_obs - 1)
    if n_pcs_effective < 2:
        raise ValueError(f"Too few dimensions for PCA: n_pcs_effective={n_pcs_effective}")

    sc.pp.pca(
        adata_harmony,
        n_comps=n_pcs_effective,
        zero_center=True,
        svd_solver=PCA_SOLVER,
        random_state=SEED,
    )
    X_pca = np.asarray(adata_harmony.obsm[PCA_KEY], dtype=np.float64)
    if X_pca.shape != (adata_harmony.n_obs, n_pcs_effective):
        raise RuntimeError(f"Unexpected PCA shape: {X_pca.shape}")
    if not np.isfinite(X_pca).all():
        raise ValueError("X_pca contains NaN or infinite values.")

    print("Running Harmony on PCA:", X_pca.shape)
    start = time.perf_counter()
    harmony_out = hm.run_harmony(
        X_pca,
        adata_harmony.obs,
        BATCH_KEY,
        theta=HARMONY_THETA,
        lamb=HARMONY_LAMBDA,
        sigma=HARMONY_SIGMA,
        nclust=HARMONY_N_CLUSTERS,
        tau=HARMONY_TAU,
        block_size=HARMONY_BLOCK_SIZE,
        max_iter_harmony=HARMONY_MAX_ITER,
        max_iter_kmeans=HARMONY_MAX_ITER_KMEANS,
        epsilon_cluster=HARMONY_EPSILON_CLUSTER,
        epsilon_harmony=HARMONY_EPSILON_HARMONY,
        alpha=HARMONY_ALPHA,
        batch_prop_cutoff=HARMONY_BATCH_PROP_CUTOFF,
        verbose=HARMONY_VERBOSE,
        random_state=SEED,
        ncores=HARMONY_NCORES,
    )
    harmony_seconds = time.perf_counter() - start

    X_harmony = np.asarray(harmony_out.Z_corr, dtype=np.float32)
    history_raw = getattr(harmony_out, "objective_harmony", None)
    history_values = [] if history_raw is None else np.asarray(history_raw, dtype=float).ravel().tolist()
    history_df = pd.DataFrame({
        "harmony_iteration": np.arange(len(history_values), dtype=int),
        "objective_harmony": np.asarray(history_values, dtype=float),
    }) if len(history_values) else pd.DataFrame()

    np.save(cached_latent_path, X_harmony)
    history_df.to_csv(cached_history_path, index=False)
    manifest_path.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")

if X_harmony.shape[0] != adata.n_obs:
    raise RuntimeError(f"Harmony row count does not match benchmark cells: {X_harmony.shape}")
if X_harmony.ndim != 2 or X_harmony.shape[1] < 2:
    raise RuntimeError(f"Unexpected Harmony latent shape: {X_harmony.shape}")
if not np.isfinite(X_harmony).all():
    raise ValueError("Harmony latent contains NaN or infinite values.")

print("Harmony latent shape:", X_harmony.shape)
print(f"Harmony integration time: {harmony_seconds:.2f} seconds")
if not history_df.empty:
    display(history_df)


## 7. Save the latent and validate cell alignment


In [ ]:
from scrarebench.latent import attach_latent

barcodes = adata_harmony.obs_names.astype(str).to_numpy()

latent_path = WORK_DIR / "Harmony_latent.npy"
barcodes_path = WORK_DIR / "Harmony_cell_barcodes.npy"
hvg_path = WORK_DIR / "Harmony_hvgs.txt"
config_path = WORK_DIR / "Harmony_config.json"
history_path = WORK_DIR / "Harmony_objective_history.csv"

np.save(latent_path, X_harmony)
np.save(barcodes_path, barcodes)
hvg_path.write_text("\n".join(map(str, adata_harmony.var_names)), encoding="utf-8")
config_payload = {**harmony_config, "harmony_seconds": harmony_seconds, "latent_dimensions": int(X_harmony.shape[1])}
config_path.write_text(json.dumps(config_payload, indent=2), encoding="utf-8")
history_df.to_csv(history_path, index=False)

alignment_report = attach_latent(
    adata,
    X_harmony,
    key=LATENT_KEY,
    latent_barcodes=barcodes,
    allow_reorder=False,
    overwrite=True,
)

print("Latent:", X_harmony.shape, X_harmony.dtype)
print("Alignment report:", alignment_report)
print("Saved latent:", latent_path)
print("Saved barcodes:", barcodes_path)

# The preprocessing-only AnnData is no longer required after latent extraction.
del adata_harmony, counts_hvg
if adata_rna is not adata:
    del adata_rna
gc.collect()


## 8. Run the scRareBench evaluation


In [ ]:
from scrarebench.evaluation import EvaluationConfig, evaluate_latent
from scrarebench.scib_backend import ScibEvaluationConfig

if RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

benchmark_config = EvaluationConfig(
    method_name=METHOD_NAME,
    representation_key=LATENT_KEY,
    label_key=LABEL_KEY,
    batch_key=BATCH_KEY,
    reference_resolution=REFERENCE_RESOLUTION,
    resolution_sweep=RESOLUTION_SWEEP,
    n_neighbors=N_NEIGHBORS,
    distance_metric=DISTANCE_METRIC,
    random_state=SEED,
    overwrite=True,
    scib=ScibEvaluationConfig(
        enabled=RUN_SCIB,
        count_layer=COUNTS_LAYER,
        n_hvg=SCIB_N_HVG,
        reference_n_pcs=SCIB_REFERENCE_N_PCS,
        n_jobs=SCIB_N_JOBS,
        progress_bar=SCIB_PROGRESS_BAR,
        include_silhouette_batch=SCIB_INCLUDE_SILHOUETTE_BATCH,
        require_backend=SCIB_REQUIRE_SUCCESS,
    ),
)

result = evaluate_latent(adata, benchmark_config, RESULTS_DIR)

print("Benchmark completed.")
if result.scib is not None:
    runtime_versions = result.scib.reference_config.get("runtime_versions", {})
    compatibility_adjustments = result.scib.reference_config.get("runtime_compatibility_adjustments", [])
    print("scIB runtime versions:", runtime_versions)
    print("scIB compatibility adjustments:", compatibility_adjustments or "none required")
print("Reference cluster key:", result.cluster_keys[REFERENCE_RESOLUTION])
print("\nPaper-style overall/subset metrics")
display(result.subset_metrics.round(5))
print("\nRare-cell summary")
display(result.rare_summary.round(5))
print("\nRare-cell per-type metrics")
display(result.rare_metrics.round(5))

if result.scib is not None:
    print("\nStandard scIB-compatible aggregate scores")
    display(result.scib.aggregate_scores.round(5))
    print("\nStandard scIB-compatible individual metrics")
    display(result.scib.metrics_long.round(5))
    print("\nMetric applicability/status")
    display(result.scib.metric_status)


## 9. Add method-specific figures and build reports


In [ ]:
from scrarebench.reporting import write_html_report, write_interactive_report, write_pdf_report

figures_dir = RESULTS_DIR / "rare_cell" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
extra_figures = []

run_config_path = RESULTS_DIR / "reproducibility" / "run_config.yaml"
run_config = yaml.safe_load(run_config_path.read_text(encoding="utf-8"))
neighbors_key = run_config["neighbors_key"]
reference_cluster_key = run_config["reference_cluster_key"]

# Record Harmony preprocessing and parameters in reproducibility metadata.
run_config["method_specific"] = {
    "implementation": "harmonypy",
    "harmonypy_version": getattr(hm, "__version__", "2.0.0"),
    "preprocessing": "GEX -> batch-aware HVG on raw counts -> normalize_total -> log1p -> optional scale -> PCA -> Harmony",
    **harmony_config,
    "integration_seconds": round(harmony_seconds, 4),
}
run_config_path.write_text(yaml.safe_dump(run_config, sort_keys=False), encoding="utf-8")

package_versions_path = RESULTS_DIR / "reproducibility" / "package_versions.txt"
if package_versions_path.exists():
    version_text = package_versions_path.read_text(encoding="utf-8")
    if "harmonypy=" not in version_text:
        with package_versions_path.open("a", encoding="utf-8") as handle:
            handle.write(f"harmonypy={getattr(hm, '__version__', '2.0.0')}\n")

if GENERATE_UMAP:
    sc.tl.umap(adata, neighbors_key=neighbors_key, random_state=SEED)
    adata.obsm["X_umap_Harmony"] = adata.obsm["X_umap"].copy()

    for color_key, title, filename, size in [
        (LABEL_KEY, "Harmony UMAP — reference cell types", "umap_harmony_cell_types.png", (13, 9)),
        (BATCH_KEY, "Harmony UMAP — batches", "umap_harmony_batches.png", (11, 8)),
    ]:
        sc.pl.embedding(
            adata,
            basis="X_umap_Harmony",
            color=color_key,
            title=title,
            frameon=False,
            show=False,
            legend_loc="right margin",
        )
        path = figures_dir / filename
        plt.gcf().set_size_inches(*size)
        plt.savefig(path, dpi=180, bbox_inches="tight")
        plt.close()
        extra_figures.append(path)

    rare_mask = adata.obs["scrarebench_is_rare"].astype(bool).to_numpy()
    rare_view = adata[rare_mask].copy()
    sc.pl.embedding(
        rare_view,
        basis="X_umap_Harmony",
        color="scrarebench_scenario",
        title="Harmony — curated rare populations across six scenarios",
        frameon=False,
        show=False,
        legend_loc="right margin",
        size=24,
    )
    path = figures_dir / "umap_harmony_rare_scenarios.png"
    plt.gcf().set_size_inches(11, 8)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    extra_figures.append(path)

if not history_df.empty and "objective_harmony" in history_df.columns:
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.plot(
        history_df["harmony_iteration"].to_numpy(dtype=int),
        history_df["objective_harmony"].to_numpy(dtype=float),
        marker="o",
    )
    ax.set_xlabel("Harmony iteration")
    ax.set_ylabel("Harmony objective")
    ax.set_title("Harmony convergence objective")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    path = figures_dir / "harmony_objective_history.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    extra_figures.append(path)

base_figures = []
if result.scib is not None:
    base_figures.append(result.scib.files["metric_plot"])
base_figures.extend([
    result.files["rare_metric_heatmap"],
    result.files["rare_precision_recall"],
    result.files["failure_counts"],
])

report_metadata = {
    "method": METHOD_NAME,
    "representation_key": LATENT_KEY,
    "n_cells": adata.n_obs,
    "n_dimensions": adata.obsm[LATENT_KEY].shape[1],
    "label_key": LABEL_KEY,
    "batch_key": BATCH_KEY,
    "reference_resolution": REFERENCE_RESOLUTION,
    "n_neighbors": N_NEIGHBORS,
    "distance_metric": DISTANCE_METRIC,
    "benchmark_seed": SEED,
    "harmonypy_version": getattr(hm, "__version__", "2.0.0"),
    "Harmony_n_hvg": N_HVG,
    "Harmony_n_pcs": N_PCS,
    "Harmony_theta": HARMONY_THETA,
    "Harmony_lambda": HARMONY_LAMBDA,
    "Harmony_max_iter": HARMONY_MAX_ITER,
    "Harmony_seconds": round(harmony_seconds, 2),
    "reference_cluster_key": reference_cluster_key,
    "cell_order_exact_match": alignment_report["exact_order_match"],
    "scib_backend": result.scib.backend if result.scib else "disabled",
    "scib_backend_version": result.scib.backend_version if result.scib else "n/a",
}

report_path = RESULTS_DIR / "report.html"
write_html_report(
    report_path,
    title="scRareBench report — Harmony on GSE194122 paper benchmark",
    metadata=report_metadata,
    global_table=result.subset_metrics,
    rare_table=result.rare_metrics,
    figure_names=base_figures + extra_figures,
    scib_metrics=result.scib.metrics_long if result.scib else None,
    scib_aggregates=result.scib.aggregate_scores if result.scib else None,
    scib_status=result.scib.metric_status if result.scib else None,
    rare_summary=result.rare_summary,
    scenario_table=result.scenario_metrics,
)

print("Self-contained report:", report_path)
print("Report size (MB):", round(report_path.stat().st_size / 1024**2, 2))

interactive_report_path = RESULTS_DIR / "interactive_report.html"
pdf_report_path = RESULTS_DIR / "summary_report.pdf"

# Register notebook-generated figures with the dashboard.
for index, figure_path in enumerate(extra_figures):
    result.files[f"notebook_figure_{index:02d}"] = figure_path

HTML_REPORT_OPTIONS = {
    "include_overview": HTML_INCLUDE_OVERVIEW,
    "include_metrics": HTML_INCLUDE_METRICS,
    "include_scib": HTML_INCLUDE_SCIB,
    "include_rare": HTML_INCLUDE_RARE,
    "include_rare_umap": HTML_INCLUDE_RARE_UMAP,
    "include_rare_heatmaps": HTML_INCLUDE_RARE_HEATMAPS,
    "include_rare_scenario_analysis": HTML_INCLUDE_RARE_SCENARIO_ANALYSIS,
    "include_umap": HTML_INCLUDE_UMAP,
    "include_sankey": HTML_INCLUDE_SANKEY,
    "include_reproducibility": HTML_INCLUDE_REPRODUCIBILITY,
    "include_static_figures": HTML_INCLUDE_STATIC_FIGURES,
    "include_cell_ids": HTML_INCLUDE_CELL_IDS,
}

if GENERATE_INTERACTIVE_REPORT:
    write_interactive_report(
        adata,
        result,
        interactive_report_path,
        representation_key=LATENT_KEY,
        label_key=LABEL_KEY,
        batch_key=BATCH_KEY,
        umap_key="X_umap_Harmony" if "X_umap_Harmony" in adata.obsm else None,
        **HTML_REPORT_OPTIONS,
    )

if GENERATE_PDF_REPORT:
    write_pdf_report(
        adata,
        result,
        pdf_report_path,
        representation_key=LATENT_KEY,
    )

print("Static HTML report:", report_path)
print("Interactive HTML report:", interactive_report_path if GENERATE_INTERACTIVE_REPORT else "disabled")
print("PDF report:", pdf_report_path if GENERATE_PDF_REPORT else "disabled")


## 10. Display and inspect generated reports


In [ ]:
display(HTML(report_path.read_text(encoding="utf-8")))

print("\nGenerated output files:")
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RESULTS_DIR))


## 11. Build and download the result bundle


In [ ]:
from scrarebench.reporting import create_report_bundle

if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if (RESULTS_DIR / "interactive_report.html").exists():
    shutil.copy2(RESULTS_DIR / "interactive_report.html", ARTIFACT_DIR / "interactive_report.html")
if (RESULTS_DIR / "summary_report.pdf").exists():
    shutil.copy2(RESULTS_DIR / "summary_report.pdf", ARTIFACT_DIR / "summary_report.pdf")

summary = {
    "method": METHOD_NAME,
    "dataset": "GSE194122 paper-main benchmark",
    "n_cells": adata.n_obs,
    "latent_shape": list(adata.obsm[LATENT_KEY].shape),
    "static_report": "benchmark_results/report.html",
    "interactive_report": "reports/interactive_report.html",
    "pdf_report": "reports/summary_report.pdf",
    "include_latent": INCLUDE_LATENT_IN_BUNDLE,
    "note": "The bundle ZIP contains the full benchmark result directory, standalone interactive HTML, PDF summary, and optionally the Harmony latent matrix plus barcodes.",
}
(ARTIFACT_DIR / "README_OUTPUT.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

archive_path = create_report_bundle(
    adata,
    result,
    WORK_DIR / "Harmony_scRareBench_GSE194122_results.zip",
    representation_key=LATENT_KEY,
    include_latent=INCLUDE_LATENT_IN_BUNDLE,
    write_interactive=GENERATE_INTERACTIVE_REPORT,
    write_pdf=GENERATE_PDF_REPORT,
    interactive_report_options=HTML_REPORT_OPTIONS,
)

print("Final ZIP:", archive_path)
print("ZIP size (MB):", round(archive_path.stat().st_size / 1024 / 1024, 2))


## Interpretation notes

- Harmony uses PCA coordinates and batch labels only; `celltype` is used only by the benchmark.
- Harmony-specific preprocessing is notebook/user code and is not part of `load_dataset()`.
- The evaluated latent is stored in `adata.obsm["X_pca_harmony"]`.
- `report.html` is the static self-contained report.
- `interactive_report.html` contains benchmark metrics, scIB-compatible outputs, rare-cell exploration, UMAP, and Sankey views.
- `summary_report.pdf` is a compact shareable summary.
- The ZIP bundle can include the latent and barcode files when enabled.
- scIB-compatible aggregate scores remain separate from rare-cell scores.
- Failure-archetype rules are diagnostic/provisional and should be sensitivity-tested before strong scientific claims.
